# 02 — Baseline Model: MFCC + SVM / Random Forest

Train and evaluate baseline classifiers on UrbanSound8K using MFCC features.

**Pipeline**: Audio clips → MFCC extraction (40 coefficients + deltas) → StandardScaler → SVM or Random Forest → 10-class classification.

**Evaluation**: Using UrbanSound8K's predefined fold 10 as test set (folds 1-8 train, fold 9 val).

In [ ]:
import sys
sys.path.append("..")

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

from src.dataset import get_fold_data, get_default_split, CLASS_NAMES
from src.audio_features import extract_mfcc_batch
from src.baseline_model import train_baseline, evaluate_baseline, save_model

%matplotlib inline

DATA_DIR = "../data/UrbanSound8K"
SR = 22050
N_MFCC = 40

## 1. Load Data and Extract Features

In [ ]:
split = get_default_split()
print(f"Train folds: {split['train']}, Val folds: {split['val']}, Test folds: {split['test']}")

# Load audio data
print("\nLoading training data...")
train_audios, y_train, train_meta = get_fold_data(DATA_DIR, split["train"], sr=SR)
print(f"  {len(train_audios)} training clips")

print("Loading validation data...")
val_audios, y_val, val_meta = get_fold_data(DATA_DIR, split["val"], sr=SR)
print(f"  {len(val_audios)} validation clips")

print("Loading test data...")
test_audios, y_test, test_meta = get_fold_data(DATA_DIR, split["test"], sr=SR)
print(f"  {len(test_audios)} test clips")

In [ ]:
# Extract MFCC features
print("Extracting MFCC features...")
X_train = extract_mfcc_batch(train_audios, sr=SR, n_mfcc=N_MFCC)
X_val = extract_mfcc_batch(val_audios, sr=SR, n_mfcc=N_MFCC)
X_test = extract_mfcc_batch(test_audios, sr=SR, n_mfcc=N_MFCC)

print(f"Feature dimensions: {X_train.shape[1]} (n_mfcc={N_MFCC} x 6 stats)")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 2. Train Random Forest Baseline

In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_model = train_baseline(X_train, y_train, model_type="rf")

# Evaluate on validation set
print("\n--- Validation Results ---")
val_results_rf = evaluate_baseline(rf_model, X_val, y_val)
print(f"Validation Accuracy: {val_results_rf['accuracy']:.4f}")
print(val_results_rf["report"])

In [ ]:
# Evaluate on test set
print("--- Test Results ---")
test_results_rf = evaluate_baseline(rf_model, X_test, y_test)
print(f"Test Accuracy: {test_results_rf['accuracy']:.4f}")
print(test_results_rf["report"])

## 3. Train SVM Baseline

In [ ]:
# Train SVM
print("Training SVM (this may take a few minutes)...")
svm_model = train_baseline(X_train, y_train, model_type="svm")

# Evaluate on test set
print("\n--- Test Results ---")
test_results_svm = evaluate_baseline(svm_model, X_test, y_test)
print(f"Test Accuracy: {test_results_svm['accuracy']:.4f}")
print(test_results_svm["report"])

## 4. Confusion Matrices

In [ ]:
target_names = [CLASS_NAMES[i] for i in range(10)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, test_results_rf["predictions"],
    display_labels=target_names, ax=axes[0],
    xticks_rotation=45, cmap="Blues"
)
axes[0].set_title(f"Random Forest (acc={test_results_rf['accuracy']:.3f})")

# SVM confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, test_results_svm["predictions"],
    display_labels=target_names, ax=axes[1],
    xticks_rotation=45, cmap="Blues"
)
axes[1].set_title(f"SVM (acc={test_results_svm['accuracy']:.3f})")

plt.tight_layout()
plt.show()

## 5. Compare Models and Save Best

In [ ]:
# Summary
print("=" * 50)
print("Baseline Model Comparison (Test Set)")
print("=" * 50)
print(f"Random Forest: {test_results_rf['accuracy']:.4f}")
print(f"SVM (RBF):     {test_results_svm['accuracy']:.4f}")

# Save the better model
best_name = "rf" if test_results_rf["accuracy"] >= test_results_svm["accuracy"] else "svm"
best_model = rf_model if best_name == "rf" else svm_model
best_acc = max(test_results_rf["accuracy"], test_results_svm["accuracy"])

os.makedirs("../models", exist_ok=True)
save_model(best_model, f"../models/baseline_{best_name}.joblib")
print(f"\nBest model ({best_name.upper()}, acc={best_acc:.4f}) saved.")

## Next Steps

- **Full 10-fold CV**: Run `src/baseline_model.py::run_kfold_cv()` for proper evaluation
- **CNN model**: Train a CNN on mel-spectrograms (future milestone)
- **Cafe recordings**: Apply trained model to field recordings from NYC cafes
- **Scoring**: Combine predictions with spatial features for study-friendliness scores